## Currency Pipeline — AWS S3 Explorer

Reads Bronze and Silver Delta Lake tables directly from S3 using PyArrow.
Credentials are loaded from `../.env`. `AWS_REGION` must be set to match your S3 bucket region.

In [1]:
import os

import pandas as pd
import pyarrow.dataset as ds
import pyarrow.fs as pafs
from dotenv import load_dotenv

load_dotenv("../.env")

KEY    = os.getenv("AWS_ACCESS_KEY_ID")
SECRET = os.getenv("AWS_SECRET_ACCESS_KEY")
BUCKET = os.getenv("AWS_S3_BUCKET")
REGION = os.getenv("AWS_REGION")

if not all([KEY, SECRET, BUCKET, REGION]):
    raise EnvironmentError("AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_S3_BUCKET and AWS_REGION must all be set in .env")

s3 = pafs.S3FileSystem(access_key=KEY, secret_key=SECRET, region=REGION)


def read_s3(path: str) -> pd.DataFrame:
    """Read a Delta Lake table from S3 as a pandas DataFrame.
    PyArrow dataset discovery reads only Parquet files and skips _delta_log automatically.
    """
    return ds.dataset(path, filesystem=s3, format="parquet").to_table().to_pandas()


def read_s3_safe(path: str) -> pd.DataFrame | None:
    """Same as read_s3 but returns None (with a message) if the prefix does not exist."""
    try:
        return read_s3(path)
    except Exception:
        print(f"No data at s3://{path}")
        return None


print(f"Connected to s3://{BUCKET} ({REGION})")

Connected to s3://tanel-dataeng-2026-943755888672-eu-north-1-an (eu-north-1)


### Bronze — currencies

In [2]:
bronze_currencies = read_s3(f"{BUCKET}/bronze/currencies")
print(f"{len(bronze_currencies):,} rows")
bronze_currencies

322 rows


,id,name,short_code,code,precision,subunit,symbol,symbol_first,decimal_mark,thousands_separator,_ingested_at,_source_file,_batch_id
0,1,UAE Dirham,AED,784,2,100,د.إ,True,.,",",2026-05-18 15:13:58.934013,https://api.currencybeacon.com/v1/currencies,20260518_151334
1,2,Afghani,AFN,971,2,100,؋,False,.,",",2026-05-18 15:13:58.934013,https://api.currencybeacon.com/v1/currencies,20260518_151334
2,3,Lek,ALL,8,2,100,L,False,.,",",2026-05-18 15:13:58.934013,https://api.currencybeacon.com/v1/currencies,20260518_151334
3,4,Armenian Dram,AMD,51,2,100,դր.,False,.,",",2026-05-18 15:13:58.934013,https://api.currencybeacon.com/v1/currencies,20260518_151334
4,5,Netherlands Antillean Guilder,ANG,532,2,100,ƒ,True,",",.,2026-05-18 15:13:58.934013,https://api.currencybeacon.com/v1/currencies,20260518_151334
...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,157,CFP Franc,XPF,953,0,1,Fr,False,.,",",2026-05-18 11:49:28.937406,https://api.currencybeacon.com/v1/currencies,20260518_114926
318,158,Yemeni Rial,YER,886,2,100,﷼,False,.,",",2026-05-18 11:49:28.937406,https://api.currencybeacon.com/v1/currencies,20260518_114926
319,159,Rand,ZAR,710,2,100,R,True,.,",",2026-05-18 11:49:28.937406,https://api.currencybeacon.com/v1/currencies,20260518_114926
320,160,Zambian Kwacha,ZMW,967,2,100,ZK,False,.,",",2026-05-18 11:49:28.937406,https://api.currencybeacon.com/v1/currencies,20260518_114926


### Bronze — rates

In [3]:
bronze_rates = read_s3(f"{BUCKET}/bronze/rates")
print(f"{len(bronze_rates):,} rows")
bronze_rates.head(20)

51,200 rows


,currency,rate,rate_date,_ingested_at,_source_file,_batch_id
0,RUB,19.7335131600,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
1,KZT,127.5875409500,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
2,OMR,0.1047370700,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
3,KRW,407.6712670400,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
4,SAR,1.0211027900,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
5,AFN,17.1393248700,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
6,GHS,3.1145137100,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
7,BDT,33.4437383400,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
8,MKD,14.4128044700,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
9,AZN,0.4629042700,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441


### Silver — currencies

In [4]:
silver_currencies = read_s3(f"{BUCKET}/silver/currencies")
print(f"{len(silver_currencies):,} rows")
silver_currencies

322 rows


,id,name,short_code,code,precision,subunit,symbol,symbol_first,decimal_mark,thousands_separator,_ingested_at,_source_file,_batch_id
0,1,UAE Dirham,AED,784,2,100,د.إ,True,.,",",2026-05-18 15:13:58.934013,https://api.currencybeacon.com/v1/currencies,20260518_151334
1,2,Afghani,AFN,971,2,100,؋,False,.,",",2026-05-18 15:13:58.934013,https://api.currencybeacon.com/v1/currencies,20260518_151334
2,3,Lek,ALL,008,2,100,L,False,.,",",2026-05-18 15:13:58.934013,https://api.currencybeacon.com/v1/currencies,20260518_151334
3,4,Armenian Dram,AMD,051,2,100,դր.,False,.,",",2026-05-18 15:13:58.934013,https://api.currencybeacon.com/v1/currencies,20260518_151334
4,5,Netherlands Antillean Guilder,ANG,532,2,100,ƒ,True,",",.,2026-05-18 15:13:58.934013,https://api.currencybeacon.com/v1/currencies,20260518_151334
...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,157,CFP Franc,XPF,953,0,1,Fr,False,.,",",2026-05-18 11:49:28.937406,https://api.currencybeacon.com/v1/currencies,20260518_114926
318,158,Yemeni Rial,YER,886,2,100,ریال,False,.,",",2026-05-18 11:49:28.937406,https://api.currencybeacon.com/v1/currencies,20260518_114926
319,159,Rand,ZAR,710,2,100,R,True,.,",",2026-05-18 11:49:28.937406,https://api.currencybeacon.com/v1/currencies,20260518_114926
320,160,Zambian Kwacha,ZMW,967,2,100,ZK,False,.,",",2026-05-18 11:49:28.937406,https://api.currencybeacon.com/v1/currencies,20260518_114926


### Silver — currencies quarantine

In [5]:
silver_currencies_quarantine = read_s3_safe(f"{BUCKET}/silver/currencies_quarantine")
if silver_currencies_quarantine is not None:
    print(f"{len(silver_currencies_quarantine):,} quarantined rows")
    display(silver_currencies_quarantine)

No data at s3://tanel-dataeng-2026-943755888672-eu-north-1-an/silver/currencies_quarantine


### Silver — rates

In [6]:
silver_rates = read_s3(f"{BUCKET}/silver/rates")
print(f"{len(silver_rates):,} rows")
silver_rates.head(20)

51,200 rows


,currency,rate,rate_date,_ingested_at,_source_file,_batch_id
0,AFN,17.1393248700,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
1,ALL,22.3620572600,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
2,AMD,100.2186668100,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
3,ANG,0.4905096700,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
4,AOA,249.4085499400,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
5,ARS,379.7473282800,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
6,AUD,0.3806674400,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
7,AWG,0.4874064000,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
8,AZN,0.4629042700,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441
9,BAM,0.4575807000,2026-05-18 15:14:24,2026-05-18 15:14:42.815691,https://api.currencybeacon.com/v1/latest,20260518_151441


### Silver — rates quarantine

In [7]:
silver_rates_quarantine = read_s3_safe(f"{BUCKET}/silver/rates_quarantine")
if silver_rates_quarantine is not None:
    print(f"{len(silver_rates_quarantine):,} quarantined rows")
    display(silver_rates_quarantine)

No data at s3://tanel-dataeng-2026-943755888672-eu-north-1-an/silver/rates_quarantine
